# Topic 4: Data Quality & Validation

> **Phase 5 → Week 8 → Topic 4**

---

## The Data Quality Problem

```
"Garbage in, garbage out." — every data engineer who shipped bad data once.

Production ETL quality checks:
  Row-level: null checks, type checks, range checks, regex checks
  Dataset-level: row count, null %, duplicate %, referential integrity
  Business rules: amount > 0, order_date ≤ today, valid status values
  Cross-source: counts match between source and destination
```

---

## Interview Cheat Sheet

**Q: How do you handle bad records in a production Spark ETL?**
> The production pattern: (1) Read with `PERMISSIVE` mode + `_corrupt_record` column to capture bad rows. (2) Split into good_records (null `_corrupt_record`) and bad_records. (3) Process good_records normally. (4) Write bad_records to a **dead letter table** (separate path/table) for investigation. (5) Alert on bad_record count > threshold. Never silently drop or blindly process bad data.

**Q: What is a dead letter queue in ETL?**
> A dead letter queue (or dead letter table) is a storage location for records that failed processing — bad schema, business rule violations, parsing errors. Instead of losing them or crashing the pipeline, failed records are written aside with an error reason column and a timestamp. An operations team reviews them, fixes the data or the rule, and re-processes. It keeps pipelines running while preserving auditability.

**Q: How do you count nulls per column efficiently?**
> `df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns])` — this is a single-pass aggregation that counts nulls in all columns simultaneously. Avoid running `df.filter(col.isNull()).count()` per column — that's N separate jobs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week8 - Data Quality") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Create a dirty dataset for demos
dirty_orders = spark.createDataFrame([
    ("O001", "C001", 250.00,  "delivered", "2024-01-15"),
    ("O002", None,   89.50,   "pending",   "2024-01-16"),  # null customer
    ("O003", "C001", -100.00, "delivered", "2024-01-17"),  # negative amount
    ("O004", "C003", None,    "cancelled", "2024-01-18"),  # null amount
    ("O005", "C002", 500.00,  "UNKNOWN",   "2024-01-19"),  # invalid status
    ("O006", "C001", 75.00,   "delivered", "2099-12-31"),  # future date
    ("O001", "C001", 250.00,  "delivered", "2024-01-15"),  # duplicate!
    ("O007", "C999", 100.00,  "delivered", "2024-01-20"),  # C999 not in customers
], ["order_id", "customer_id", "amount", "status", "order_date"])

print("Dirty dataset:")
dirty_orders.show()

## Part 1: Data Profiling — Know Your Data Before You Process It

In [ ]:
# 1. Row count and basic stats
total_rows = dirty_orders.count()
print(f"Total rows: {total_rows}")

# 2. Null count per column (one-pass aggregation)
null_counts = dirty_orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in dirty_orders.columns
])
print("\nNull counts per column:")
null_counts.show()

# 3. Null percentage per column
null_pct = dirty_orders.select([
    (F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100).alias(f"{c}_null_pct")
    for c in dirty_orders.columns
])
print("Null percentage per column:")
null_pct.show()

# 4. Duplicate count
dup_count = dirty_orders.count() - dirty_orders.dropDuplicates(["order_id"]).count()
print(f"Duplicate order_ids: {dup_count}")

In [ ]:
# Built-in profiling: describe() + summary()
print("describe() — basic statistics:")
dirty_orders.describe("amount").show()

# summary() adds percentiles
print("summary() — extended statistics:")
dirty_orders.select("amount").summary("count", "min", "25%", "50%", "75%", "max").show()

# Value distribution for categorical columns
print("Status value distribution:")
dirty_orders.groupBy("status") \
            .agg(F.count("*").alias("count"),
                 (F.count("*") / total_rows * 100).alias("pct")) \
            .orderBy("count", ascending=False) \
            .show()

## Part 2: Row-Level Validation Rules

In [ ]:
import functools
# Define validation rules as column expressions → True = valid, False = invalid
VALID_STATUSES = ["delivered", "pending", "cancelled", "processing"]

validation_rules = {
    "order_id_not_null":    F.col("order_id").isNotNull(),
    "customer_id_not_null": F.col("customer_id").isNotNull(),
    "amount_positive":      F.col("amount") > 0,
    "amount_not_null":      F.col("amount").isNotNull(),
    "valid_status":         F.col("status").isin(VALID_STATUSES),
    "date_not_future":      F.to_date("order_date") <= F.current_date(),
}

# Apply all rules as columns
validated = dirty_orders
for rule_name, rule_expr in validation_rules.items():
    validated = validated.withColumn(rule_name, rule_expr)

# Overall valid flag: all rules must pass
validated = validated.withColumn(
    "_is_valid",
    functools.reduce(lambda a, b: a & b, [F.col(r) for r in validation_rules])
)

print("Validation results:")
validated.select(["order_id", "amount", "status", "order_date"] + 
                 list(validation_rules.keys()) + ["_is_valid"]).show(truncate=False)

In [ ]:
# Validation summary report
print("Validation Summary:")
print(f"  Total rows:    {validated.count()}")
print(f"  Valid rows:    {validated.filter(F.col('_is_valid')).count()}")
print(f"  Invalid rows:  {validated.filter(~F.col('_is_valid')).count()}")
print()

for rule_name in validation_rules:
    failing = validated.filter(~F.col(rule_name)).count()
    if failing > 0:
        print(f"  ✗ {rule_name}: {failing} violations")
    else:
        print(f"  ✓ {rule_name}: OK")

## Part 3: Dead Letter Pattern

In [ ]:
# Production pattern: good records → main pipeline, bad records → dead letter table

# Add a human-readable error reason column for bad records
error_reasons = []
for rule_name in validation_rules:
    error_reasons.append(
        F.when(~F.col(rule_name), F.lit(rule_name)).otherwise(None)
    )

# Collect all failing rule names for each row
with_reasons = validated.withColumn(
    "_error_reasons",
    F.array_compact(F.array(*error_reasons))  # array of failed rule names
).withColumn(
    "_error_count", F.size("_error_reasons")
).withColumn(
    "_processed_at", F.current_timestamp()
)

# Split
base_cols = dirty_orders.columns
good_records = with_reasons.filter(F.col("_is_valid")).select(base_cols)
dead_letter  = with_reasons.filter(~F.col("_is_valid")) \
                            .select(base_cols + ["_error_reasons", "_error_count", "_processed_at"])

print(f"Good records ({good_records.count()}):")
good_records.show()

print(f"Dead letter records ({dead_letter.count()}):")
dead_letter.select("order_id", "_error_reasons", "_error_count").show(truncate=False)

# Write dead letter table
dead_letter.write.mode("append").partitionBy("_processed_at") \
           .json("/tmp/dead_letter_orders")
print("Dead letter records written to /tmp/dead_letter_orders")

## Part 4: Dataset-Level Quality Checks

In [ ]:
# Dataset-level checks: things that are true about the WHOLE dataset

def check_row_count(df, expected_min, expected_max=None, label=""):
    count = df.count()
    ok = count >= expected_min and (expected_max is None or count <= expected_max)
    status = "✓" if ok else "✗"
    print(f"  {status} {label} row count: {count} (expected >= {expected_min})")
    return ok

def check_null_pct(df, col_name, max_null_pct, label=""):
    total = df.count()
    nulls = df.filter(F.col(col_name).isNull()).count()
    pct = (nulls / total * 100) if total > 0 else 0
    ok = pct <= max_null_pct
    status = "✓" if ok else "✗"
    print(f"  {status} {label} null%: {pct:.1f}% (max allowed {max_null_pct}%)")
    return ok

def check_uniqueness(df, key_cols, label=""):
    total = df.count()
    unique = df.dropDuplicates(key_cols).count()
    ok = total == unique
    status = "✓" if ok else "✗"
    print(f"  {status} {label} uniqueness: {unique}/{total} unique ({total-unique} duplicates)")
    return ok

def check_referential_integrity(df, fk_col, ref_df, pk_col, label=""):
    orphans = df.join(ref_df, df[fk_col] == ref_df[pk_col], "left_anti").count()
    ok = orphans == 0
    status = "✓" if ok else "✗"
    print(f"  {status} {label} referential integrity: {orphans} orphan rows")
    return ok

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)

print("Dataset-Level Quality Report:")
print("=" * 50)
check_row_count(good_records, expected_min=1, label="orders")
check_null_pct(good_records, "customer_id", max_null_pct=5.0, label="customer_id")
check_uniqueness(good_records, ["order_id"], label="order_id")
check_referential_integrity(good_records, "customer_id", customers, "customer_id", label="orders→customers")

## Part 5: Deduplication Strategies

In [ ]:
# Deduplication strategies

df_with_dupes = spark.createDataFrame([
    ("O001", "2024-01-15 10:00", 250.0, "pending"),
    ("O001", "2024-01-15 10:05", 250.0, "delivered"),  # updated later
    ("O002", "2024-01-16 09:00",  89.5, "delivered"),
    ("O002", "2024-01-16 09:00",  89.5, "delivered"),  # exact duplicate
    ("O003", "2024-01-17 11:00", 100.0, "cancelled"),
], ["order_id", "updated_at", "amount", "status"])

# Strategy 1: dropDuplicates — keeps first seen (arbitrary order!)
dedup_simple = df_with_dupes.dropDuplicates(["order_id"])
print("dropDuplicates (arbitrary winner):")
dedup_simple.show()

# Strategy 2: Keep LATEST record per key (deterministic — prefer this!)
from pyspark.sql.window import Window

w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
dedup_latest = df_with_dupes \
    .withColumn("_rn", F.row_number().over(w)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

print("Keep latest (row_number — deterministic winner):")
dedup_latest.show()

## Exercises

1. Write a function `data_profile(df)` that returns: total rows, null count per column, distinct count per column, and min/max for numeric columns — all in a single Spark action.
2. Add a `check_range(df, col, min_val, max_val)` function to the quality check suite. Apply it to `amount` to ensure values are between 0 and 100,000.
3. Read the orders CSV and apply the dead letter pattern: capture rows where `amount` is null or negative to a dead letter output.
4. What is the difference between `dropDuplicates()` and keeping the latest record via `row_number()`? When would you use each?
5. A production orders table has 1M rows. You receive a 50k-row update batch. Some are updates to existing orders, some are new. How do you merge these without creating duplicates?

In [ ]:
# Exercise 1: data_profile function
def data_profile(df):
    """Single-pass profile of a DataFrame."""
    n = df.count()
    numeric_cols = [f.name for f in df.schema.fields
                    if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType, DecimalType))]

    agg_exprs = [F.count("*").alias("_total_rows")]
    for c in df.columns:
        agg_exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls"))
        agg_exprs.append(F.approx_count_distinct(c).alias(f"{c}_distinct"))
    for c in numeric_cols:
        agg_exprs.append(F.min(c).alias(f"{c}_min"))
        agg_exprs.append(F.max(c).alias(f"{c}_max"))

    return df.agg(*agg_exprs)  # single pass!

orders = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
print("Profile of orders dataset (one Spark action):")
data_profile(orders).show(vertical=True, truncate=False)